# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baselsalah342-max/flyrank_intern/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Rule in plain words (3 sentences):**

> A page is worth refreshing if it used to get meaningful traffic, it has gone stale (not updated in 90+ days), and it sits where a refresh could matter. More stale traffic at risk ranks higher, because fixing it saves more impressions. If it is not stale or has no visibility, leave it alone.

**Signals the rule leans on:**
1. **Staleness** — `days_since_last_update` / `freshness_tier` (behind FlyRank refresh flags).
2. **Volume** — `impressions_90d` / `impression_tier` (behind FlyRank quick-win logic: high/moderate volume pages where fixing saves the most).

**Reason codes (ONE per row):**
- `stale_moderate_visible` — stale (>=90 days) AND moderate visibility (100-2999 impressions) — the queue's candidates.
- `low_priority` — everything else (not stale or not visible enough).

**Action labels:**
- `refresh` when score > 0, else `monitor`.

**Why these thresholds:** staleness >=90 matches the bucket boundary where decline rate jumps; moderate visibility (100-2999) is where decline is highest and impressions are enough to matter but not so high that the rate is already diluted. Score is `stale * moderate * impressions_90d` — readable, no fitted weights, ranks by traffic at stake.

**Label definition (no leakage):** `is_declining = (trend_direction == "down")`. Features never use `trend_direction` or `trend_pct`.

In [1]:
import pandas as pd, numpy as np, pathlib, json, os, sys

# Load starter slice — robust path search for repo vs Colab
candidates = [
    "data/raw/content_refresh_anonymized.csv",
    "../..//data/raw/content_refresh_anonymized.csv",
    "/content/flyrank_intern/data/raw/content_refresh_anonymized.csv",
    str(pathlib.Path.cwd()/"data/raw/content_refresh_anonymized.csv"),
]
# also try walking up from notebook location
try:
    nb_dir = pathlib.Path.cwd()
    for parent in [nb_dir, nb_dir.parent, nb_dir.parent.parent]:
        cand = parent/"data/raw/content_refresh_anonymized.csv"
        candidates.append(str(cand))
except Exception:
    pass
path = None
for cand in candidates:
    if os.path.exists(cand):
        path = cand
        break
# fallback: try git root via git rev-parse
if path is None:
    import subprocess
    try:
        root = subprocess.check_output(["git","rev-parse","--show-toplevel"], text=True).strip()
        cand = pathlib.Path(root)/"data/raw/content_refresh_anonymized.csv"
        if cand.exists():
            path = str(cand)
    except Exception:
        pass
if path is None:
    raise FileNotFoundError(f"Could not find content_refresh_anonymized.csv. Tried: {candidates}")
print(f"Loading: {path}")
df = pd.read_csv(path)

df["is_declining"] = (df["trend_direction"] == "down").astype(int)
base_rate = df["is_declining"].mean()
print(f"Rows: {len(df):,}  |  Base rate declining: {base_rate:.4f} (n={len(df):,})")
print(df["trend_direction"].value_counts().to_string())

# Helper to print bucket table with n
def bucket_table(group_col, label_col="is_declining"):
    g = df.groupby(group_col, observed=True)[label_col].agg(mean="mean", n="count", declining_sum="sum")
    g["mean"] = g["mean"].round(4)
    # sort by bucket order where possible
    return g.sort_values("mean", ascending=False)

# --- Signal 1: Staleness (flag-linked: refresh flags) — freshness_tier ---
print("\n=== Signal 1: Staleness (freshness_tier) — flag-linked (refresh flags) ===")
t1 = bucket_table("freshness_tier")
print(t1.to_string())
print("\ndays_since_last_update describe:")
print(df["days_since_last_update"].describe().round(2).to_string())
print("\nVerdict staleness: CONFIRMED")
print("Reason: 91-180 days stale has 61.1% declining (n=9,171) vs 51.1% for 0-30 (n=20,480). Direction matches refresh-flag assumption: staler -> more likely declining. 181+ drops to 47.1% (n=174) but n is tiny.")

# --- Signal 2: Volume (flag-linked: quick-win = volume behind CTR-fix logic) — impression_tier ---
print("\n=== Signal 2: Volume (impression_tier) — flag-linked (quick-win) ===")
t2 = bucket_table("impression_tier")
print(t2.to_string())
print("\nImpressions_90d numeric buckets:")
df["_imp_bucket"] = pd.cut(df["impressions_90d"], bins=[0,100,500,3000,30000,600000], labels=["<100","100-500","500-3k","3k-30k","30k+"], include_lowest=True)
t2b = bucket_table("_imp_bucket")
print(t2b.to_string())
print("\nVerdict volume: MIXED")
print("Reason: Moderate (61.5%, n=10,469) and 500-3k (62.1%, n=8,432) are highest; low (<100) is 38.9% (n=8,006) and excellent/30k+ is 46.2% (n=1,078) are lowest. Not monotonic — quick-win assumption 'more volume = more risk' only holds up to moderate, then inverts. So MIXED, still flag-linked and saves rule.")


Loading: ../..//data/raw/content_refresh_anonymized.csv
Rows: 30,000  |  Base rate declining: 0.5421 (n=30,000)
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152

=== Signal 1: Staleness (freshness_tier) — flag-linked (refresh flags) ===
                  mean      n  declining_sum
freshness_tier                              
91-180          0.6111   9171           5604
31-90           0.5886    175            103
0-30            0.5114  20480          10473
181+            0.4713    174             82

days_since_last_update describe:


count    30000.00
mean        46.10
std         42.08
min          1.00
25%         20.00
50%         20.00
75%        104.00
max        373.00

Verdict staleness: CONFIRMED
Reason: 91-180 days stale has 61.1% declining (n=9,171) vs 51.1% for 0-30 (n=20,480). Direction matches refresh-flag assumption: staler -> more likely declining. 181+ drops to 47.1% (n=174) but n is tiny.

=== Signal 2: Volume (impression_tier) — flag-linked (quick-win) ===
                   mean      n  declining_sum
impression_tier                              
moderate         0.6147  10469           6435
good             0.5861   7205           4223
excellent        0.4620   1078            498
low              0.4539  11248           5106

Impressions_90d numeric buckets:
               mean     n  declining_sum
_imp_bucket                             
500-3k       0.6208  8432           5235
100-500      0.6043  5279           3190
3k-30k       0.5861  7205           4223
30k+         0.4620  1078           

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

**Score formula (transparent, no fitted weights):**
```python
stale    = (days_since_last_update >= 90).astype(int)   # 91-180 bucket and up
moderate = ((impressions_90d >= 100) & (impressions_90d < 3000)).astype(int)
score    = stale * moderate * impressions_90d            # ranks by traffic at stake among stale+visible
reason_code = "stale_moderate_visible" if score>0 else "low_priority"
action      = "refresh" if score>0 else "monitor"
```
Only `days_since_last_update` and `impressions_90d` — no `trend_pct`, no `trend_direction`, no future windows.

In [2]:
import pathlib

# --- Encode rule (ONE score, ONE reason code, ONE action) ---
stale = (df["days_since_last_update"] >= 90).astype(int)
moderate = ((df["impressions_90d"] >= 100) & (df["impressions_90d"] < 3000)).astype(int)
# Also keep visible for clarity
df["stale"] = stale
df["moderate"] = moderate
df["score"] = stale * moderate * df["impressions_90d"]
df["reason_code"] = np.where(df["score"] > 0, "stale_moderate_visible", "low_priority")
df["action"] = np.where(df["score"] > 0, "refresh", "monitor")

# Rank
ranked = df.sort_values("score", ascending=False).reset_index(drop=True)
ranked["rank"] = ranked.index + 1

# Precision@K helper
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base = df["is_declining"].mean()
print(f"Base rate: {base:.4f}")
for k in [10,20,50,100,500,1000]:
    p = precision_at_k(ranked["score"], ranked["is_declining"], k)
    print(f"precision@{k:>4}: {p:.3f}  (n={k})")
# Dummy floor: random / majority
np.random.seed(0)
rand_p50 = precision_at_k(np.random.rand(len(df)), df["is_declining"], 50)
print(f"random precision@50: {rand_p50:.3f} (sanity: ~ base rate)")
print(f"Candidates (score>0): {(df['score']>0).sum():,} / {len(df):,}")

# Prepare CSV — include minimal required + context cols, sorted by rank
out_cols = ["rank","content_id","client_id","score","reason_code","action",
            "days_since_last_update","freshness_tier","impressions_90d","impression_tier",
            "avg_position","position_tier","ctr","is_declining"]
# Ensure no private data: content_id/client_id are pseudonyms already, no URLs/names
export = ranked[out_cols].copy()

# Write from notebook (regenerates on every run)
import subprocess
# Resolve output dir robustly
for cand_root in [pathlib.Path.cwd(), pathlib.Path.cwd().parent, pathlib.Path.cwd().parent.parent]:
    if (cand_root/"data/raw/content_refresh_anonymized.csv").exists():
        repo_root = cand_root
        break
else:
    try:
        repo_root = pathlib.Path(subprocess.check_output(["git","rev-parse","--show-toplevel"], text=True).strip())
    except Exception:
        repo_root = pathlib.Path.cwd()
out_path = repo_root/"work/outputs/baseline_action_score.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
export.to_csv(out_path, index=False)
print(f"\nWrote {out_path} with {len(export):,} rows, {export['score'].gt(0).sum():,} scored >0")
print(export.head(10).to_string(index=False))

# Save metrics JSON (committed as receipt)
metrics = {
    "base_rate": round(float(base), 4),
    "candidates": int((df["score"]>0).sum()),
    "precision_at_k": {str(k): round(float(precision_at_k(ranked["score"], ranked["is_declining"], k)),4) for k in [10,20,50,100,500,1000]},
    "rule": "stale(>=90) * moderate(100<=imp<3000) * impressions_90d",
    "reason_codes": ["stale_moderate_visible","low_priority"],
    "n_rows": len(df)
}
with open(repo_root/"work/outputs/baseline_metrics.json","w") as f:
    json.dump(metrics, f, indent=2)
print("\nMetrics:", json.dumps(metrics, indent=2))


Base rate: 0.5421
precision@  10: 0.800  (n=10)
precision@  20: 0.850  (n=20)
precision@  50: 0.740  (n=50)
precision@ 100: 0.680  (n=100)
precision@ 500: 0.708  (n=500)
precision@1000: 0.671  (n=1000)
random precision@50: 0.540 (sanity: ~ base rate)
Candidates (score>0): 4,595 / 30,000



Wrote /home/basel/fly rank assignments/machine_learning/flyrank_intern/work/outputs/baseline_action_score.csv with 30,000 rows, 4,595 scored >0
 rank           content_id         client_id  score            reason_code  action  days_since_last_update freshness_tier  impressions_90d impression_tier  avg_position position_tier  ctr  is_declining
    1 content_19882bd6373d client_8527a891e2   2999 stale_moderate_visible refresh                     104         91-180             2999        moderate           9.0        page_1 0.30             1
    2 content_cfbb20d73437 client_6208ef0f77   2999 stale_moderate_visible refresh                     104         91-180             2999        moderate          31.9      page_3_5 0.03             1
    3 content_4302c4925c0f client_3fdba35f04   2998 stale_moderate_visible refresh                     104         91-180             2998        moderate          18.7      striking 0.10             1
    4 content_3344bfd6994d client_19581e27de   

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

All top 20 share `reason_code=stale_moderate_visible` and `action=refresh` — they are stale (104-106 days) and moderate visibility (~2,950-2,999 impressions). Rank is by impressions (more at stake first). Below is one line per row of the top 10 (top 20 in code), plus table.

**Top-10 one-line reviews:**

1. `content_19882bd6373d` — refresh / stale_moderate_visible — 2999 imp, 104d stale, page_1 — confidence high but CTR 0.30 is not alarming — **wrong if**: traffic stable despite staleness, or mid-season seasonality not decline.
2. `content_cfbb20d73437` — refresh / stale_moderate_visible — 2999 imp, 104d, page_3_5 (pos ~20+) — low CTR 0.03 — **wrong if**: already ranking poorly for low-volume keyword, no click potential.
3. `content_4302c4925c0f` — refresh / stale_moderate_visible — 2998 imp, striking 11-20, CTR 0.10 — **wrong if**: position is the real cause and content refresh won't move it without backlinks.
4. `content_3344bfd6994d` — refresh / stale_moderate_visible — 2997 imp, CTR 1.07 relatively high — **wrong if**: this is actually healthy (stable label), rank wasted — shows rule doesn't check CTR.
5. `content_7fde62f5a97f` — refresh / stale_moderate_visible — 2993 imp, CTR 0.70 — **wrong if**: clicks already above average, decline may be impressions seasonality not content decay.
6. `content_ccf887ee3581` — refresh / stale_moderate_visible — 2993 imp, CTR 0.03 very low — **wrong if**: impressions are inflated by irrelevant queries, refresh won't fix intent mismatch.
7. `content_51bb0bff5aed` — refresh / stale_moderate_visible — 2993 imp, striking — **wrong if**: SERP feature stole clicks, not content staleness.
8. `content_b725a6ce11a9` — refresh / stale_moderate_visible — 2988 imp, page_3_5 — **wrong if**: deep position (~30-50), needs keyword strategy not refresh.
9. `content_a0e08775e954` — refresh / stale_moderate_visible — 2986 imp, CTR 0.10 — **wrong if**: actually UP trend (label 0), rule false positive — picked on staleness alone.
10. `content_a34aff7561ae` — refresh / stale_moderate_visible — 2986 imp, striking — **wrong if**: small-sample CTR noise, or single client over-representation.

Confidence overall: moderate-high for 1-3,6-7 (stale + striking/low-CTR aligns), lower for 4,9 (labels show not declining — weak picks proving rule is honest but imperfect).

In [3]:
# Display top 20 with context for hand review
cols = ["rank","content_id","score","reason_code","action","freshness_tier","days_since_last_update","impression_tier","impressions_90d","position_tier","avg_position","ctr","trend_direction","is_declining"]
# trend_direction shown for validation only, never used as feature
top20 = ranked[cols].head(20)
# Pretty print
pd.set_option("display.max_columns", None)
print(top20.to_string(index=False))

# Also save top20 for report
top20.to_csv(repo_root/"work/outputs/baseline_top20.csv", index=False)
print("\nSaved work/outputs/baseline_top20.csv")


 rank           content_id  score            reason_code  action freshness_tier  days_since_last_update impression_tier  impressions_90d position_tier  avg_position  ctr trend_direction  is_declining
    1 content_19882bd6373d   2999 stale_moderate_visible refresh         91-180                     104        moderate             2999        page_1           9.0 0.30            down             1
    2 content_cfbb20d73437   2999 stale_moderate_visible refresh         91-180                     104        moderate             2999      page_3_5          31.9 0.03            down             1
    3 content_4302c4925c0f   2998 stale_moderate_visible refresh         91-180                     104        moderate             2998      striking          18.7 0.10            down             1
    4 content_3344bfd6994d   2997 stale_moderate_visible refresh         91-180                     104        moderate             2997        page_1           3.1 1.07          stable             0


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks in top 20:** 3 of top 20 are NOT declining (stable/up) — e.g., rank 4 (`content_3344bfd6994d`, stable, CTR 1.07), rank 9 (`content_a0e08775e954`, up), rank ~14 stable. They share the same stale+moderate pattern but CTR or position doesn't fit the decline stereotype. This is expected: the rule over-triggers on staleness+volume without checking CTR or position change, so healthy pages slip in. If I looked harder, I'd also question tying rank strictly to raw impressions within moderate band — a 2999 vs 2900 difference is noise, not priority.

**Leakage check:**
- Features used: `days_since_last_update`, `impressions_90d` only (derived `freshness_tier`/`impression_tier` for buckets, not for score except moderate definition).
- NOT used: `trend_pct`, `trend_direction`, `is_declining_label`, `impressions_last_30d`/`_prev_30d` in score (they are trend inputs), no future window.
- Rate/position not in score, so no CTR-vs-position leakage trick.
- IDs pseudonymous, used only for grouping/output, never as numeric feature.
- No `target` derived from future 30d — label is defined from trend_pct which we never touch.

In [4]:
# Leakage audit: list every column touching the score/feature set
allowed = {"days_since_last_update","impressions_90d","score","stale","moderate","reason_code","action","rank"}
# The rule touches only those two raw cols
print("Columns used in score: days_since_last_update, impressions_90d")
print("Forbidden (leakage) columns NOT used: trend_pct, trend_direction, is_declining, impressions_last_30d, impressions_prev_30d, clicks_last_30d etc.")
# Verify CSV doesn't contain leakage cols as features (it has them for context but score wasn't built from them)
print("\nCSV columns:", export.columns.tolist())
# Weak picks summary
top20_declining = top20["is_declining"].sum()
print(f"\nTop 20 precision: {top20_declining}/20 = {top20_declining/20:.2f}")
print(f"Top 50 precision: {precision_at_k(ranked['score'], ranked['is_declining'], 50):.2f} (vs base {base:.2f})")
# If no weak picks found, flag
if top20_declining == 20:
    print("WARNING: no weak picks — look harder")
else:
    print(f"Found {20-top20_declining} weak picks in top 20 — honest baseline.")

# Confirm no trend_pct in score correlation by construction
print("\nScore corr with is_declining:", df["score"].corr(df["is_declining"]).round(4))
# Check that score doesn't use avg_position or ctr
print("Score unique values:", df["score"].nunique(), "non-zero:", (df["score"]>0).sum())


Columns used in score: days_since_last_update, impressions_90d
Forbidden (leakage) columns NOT used: trend_pct, trend_direction, is_declining, impressions_last_30d, impressions_prev_30d, clicks_last_30d etc.

CSV columns: ['rank', 'content_id', 'client_id', 'score', 'reason_code', 'action', 'days_since_last_update', 'freshness_tier', 'impressions_90d', 'impression_tier', 'avg_position', 'position_tier', 'ctr', 'is_declining']

Top 20 precision: 17/20 = 0.85
Top 50 precision: 0.74 (vs base 0.54)
Found 3 weak picks in top 20 — honest baseline.

Score corr with is_declining: 0.0819
Score unique values: 2051 non-zero: 4595


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Lane:** Refresh-risk queue (staleness + visibility). Lanes lock this week — keeping this lane. Week-5 model must beat precision@50 = 0.74 (vs base 0.54).